# Cyberbullying Detection — Kaggle Dataset
## Lightweight 2-Seed Robustness Check: other_cyberbullying Class
Your seed=2042 run already showed a highly significant, class-restricted McNemar's result on `other_cyberbullying` (p<0.0001, 217 vs 28 disagreements). This notebook re-runs the full pipeline with **2 additional seeds** (7, 123) to check whether that specific finding is robust — not a repeat of your original run, to save GPU time.

For each new seed, this notebook trains both models, then runs the class-restricted McNemar's test on `other_cyberbullying` immediately, so you get the one number that matters most without extra steps.

**Runtime:** ~2 full training runs (baseline + attention) per seed, same scale as your original Kaggle run (~4-6 hours per seed with early stopping). Plan for roughly 8-12 hours total across both seeds, or run them in separate Kaggle sessions if needed.

## 1. Setup

In [ ]:
!pip install -q emoji contractions imbalanced-learn statsmodels
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import string
import time
import random

import emoji
import contractions
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import RandomOverSampler
from statsmodels.stats.contingency_tables import mcnemar

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertModel, BertTokenizer
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

sns.set_style("whitegrid")
plt.rc("figure", autolayout=True)
plt.rc("axes", labelweight="bold", labelsize="large", titleweight="bold", titlepad=10)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 2. Load + clean Kaggle dataset (done ONCE)

In [ ]:
base_path = '/kaggle/input/cyberbullying-classification/'

df = pd.read_csv(base_path + 'cyberbullying_tweets.csv', encoding='latin1')
df = df.rename(columns={'tweet_text': 'text', 'cyberbullying_type': 'sentiment'})
df = df[~df.duplicated()]
print(df.shape)

In [ ]:
def strip_emoji(text):
    if not isinstance(text, str):
        text = str(text)
    emoji_pattern = re.compile(
        "[\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FA6F"
        "\U0001FA70-\U0001FAFF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

def strip_all_entities(text):
    if not isinstance(text, str):
        text = str(text)
    text = re.sub(r'\r|\n', ' ', text.lower())
    text = re.sub(r"(?:\@|https?\://)\S+", "", text)
    text = re.sub(r'[^\x00-\x7f]', '', text)
    table = str.maketrans('', '', string.punctuation)
    text = text.translate(table)
    text = ' '.join(word for word in text.split() if word not in stop_words)
    return text

def clean_hashtags(tweet):
    if not isinstance(tweet, str):
        tweet = str(tweet)
    new_tweet = re.sub(r'(\s+#[\w-]+)+\s*$', '', tweet).strip()
    new_tweet = re.sub(r'#([\w-]+)', r'\1', new_tweet).strip()
    return new_tweet

def filter_chars(text):
    if not isinstance(text, str):
        text = str(text)
    return ' '.join('' if ('$' in word) or ('&' in word) else word for word in text.split())

def remove_mult_spaces(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(r"\s\s+", " ", text)

def expand_contractions(text):
    if not isinstance(text, str):
        text = str(text)
    return contractions.fix(text)

def remove_numbers(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(r'\d+', '', text)

def lemmatize(text):
    if not isinstance(text, str):
        text = str(text)
    words = word_tokenize(text)
    return ' '.join(lemmatizer.lemmatize(w) for w in words)

def remove_short_words(text, min_len=2):
    if not isinstance(text, str):
        text = str(text)
    return ' '.join(w for w in text.split() if len(w) >= min_len)

def replace_elongated_words(text):
    if not isinstance(text, str):
        text = str(text)
    regex_pattern = r'\b(\w+)((\w)\3{2,})(\w*)\b'
    return re.sub(regex_pattern, r'\1\3\4', text)

def remove_repeated_punctuation(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(r'[\?\.\!]+(?=[\?\.\!])', '', text)

def remove_extra_whitespace(text):
    if not isinstance(text, str):
        text = str(text)
    return ' '.join(text.split())

def remove_url_shorteners(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(
        r'(?:http[s]?://)?(?:www\.)?(?:bit\.ly|goo\.gl|t\.co|tinyurl\.com|tr\.im|is\.gd|'
        r'cli\.gs|u\.nu|url\.ie|tiny\.cc|alturl\.com|ow\.ly|bit\.do|adoro\.to)\S+', '', text)

def remove_spaces_tweets(tweet):
    if not isinstance(tweet, str):
        tweet = str(tweet)
    return tweet.strip()

def remove_short_tweets(tweet, min_words=3):
    if not isinstance(tweet, str):
        tweet = str(tweet)
    words = tweet.split()
    return tweet if len(words) >= min_words else ""

def clean_tweet(tweet):
    if not isinstance(tweet, str):
        tweet = str(tweet)
    tweet = strip_emoji(tweet)
    tweet = expand_contractions(tweet)
    tweet = strip_all_entities(tweet)
    tweet = clean_hashtags(tweet)
    tweet = filter_chars(tweet)
    tweet = remove_mult_spaces(tweet)
    tweet = remove_numbers(tweet)
    tweet = lemmatize(tweet)
    tweet = remove_short_words(tweet)
    tweet = replace_elongated_words(tweet)
    tweet = remove_repeated_punctuation(tweet)
    tweet = remove_extra_whitespace(tweet)
    tweet = remove_url_shorteners(tweet)
    tweet = remove_spaces_tweets(tweet)
    tweet = ' '.join(tweet.split())
    return tweet

df['text_clean'] = [clean_tweet(t) for t in df['text']]
df.drop_duplicates('text_clean', inplace=True)
df = df[df['text_clean'].str.len() > 0]

sentiment = ["not_cyberbullying", "gender", "religion", "age", "ethnicity", "other_cyberbullying"]
TARGET_CLASS_IDX = sentiment.index("other_cyberbullying")

df['text_len'] = [len(t.split()) for t in df['text_clean']]
df = df[df['text_len'] < df['text_len'].quantile(0.995)]
label_map = {name: i for i, name in enumerate(sentiment)}
df['sentiment'] = df['sentiment'].replace(label_map)
print(f"Final dataset size: {len(df)}")

X_all = df['text_clean'].values
y_all = df['sentiment'].values

## 3. Model definitions (identical to your validated Kaggle notebook)

In [ ]:
N_OUTPUT = 6

class Bert_Classifier_CLS(nn.Module):
    def __init__(self, freeze_bert=False):
        super().__init__()
        n_hidden = 50
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.classifier = nn.Sequential(
            nn.Linear(768, n_hidden), nn.ReLU(), nn.Linear(n_hidden, N_OUTPUT)
        )
        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_vec = outputs[0][:, 0, :]
        return self.classifier(cls_vec)


class Bert_Classifier_AttentionPool(nn.Module):
    def __init__(self, freeze_bert=False):
        super().__init__()
        n_hidden = 128
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.attention_weights = nn.Sequential(
            nn.Linear(768, 128), nn.Tanh(), nn.Linear(128, 1)
        )
        self.classifier = nn.Sequential(
            nn.Linear(768, n_hidden), nn.ReLU(), nn.Dropout(0.3), nn.Linear(n_hidden, N_OUTPUT)
        )
        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs[0]
        attn_scores = self.attention_weights(last_hidden_state).squeeze(-1)
        attn_scores = attn_scores.masked_fill(attention_mask == 0, -1e9)
        attn_probs = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled_output = torch.sum(last_hidden_state * attn_probs, dim=1)
        return self.classifier(pooled_output)

## 4. Tokenizer (loaded once, reused across seeds)

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
MAX_LEN = 128

def bert_tokenizer(data):
    input_ids, attention_masks = [], []
    for sent in data:
        encoded_sent = tokenizer(
            sent, add_special_tokens=True, max_length=MAX_LEN,
            padding='max_length', truncation=True, return_attention_mask=True
        )
        input_ids.append(encoded_sent['input_ids'])
        attention_masks.append(encoded_sent['attention_mask'])
    return torch.tensor(input_ids), torch.tensor(attention_masks)

## 5. Shared training / evaluation functions

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def initialize_model(model_class, train_dataloader, epochs=10):
    model = model_class(freeze_bert=False)
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=5e-5, eps=1e-8)
    total_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
    return model, optimizer, scheduler


def bert_train(model, optimizer, scheduler, train_dataloader, val_dataloader, epochs=10, patience=3, name=""):
    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0

    for epoch_i in range(epochs):
        t0_epoch = time.time()
        total_loss = 0
        model.train()
        for step, batch in enumerate(train_dataloader):
            b_input_ids, b_attn_mask, b_labels = tuple(t.to(device) for t in batch)
            model.zero_grad()
            logits = model(b_input_ids, b_attn_mask)
            loss = loss_fn(logits, b_labels)
            total_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
        avg_train_loss = total_loss / len(train_dataloader)

        model.eval()
        val_accuracy, val_loss = [], []
        for batch in val_dataloader:
            b_input_ids, b_attn_mask, b_labels = tuple(t.to(device) for t in batch)
            with torch.no_grad():
                logits = model(b_input_ids, b_attn_mask)
            loss = loss_fn(logits, b_labels)
            val_loss.append(loss.item())
            preds = torch.argmax(logits, dim=1).flatten()
            val_accuracy.append((preds == b_labels).cpu().numpy().mean() * 100)
        val_loss = np.mean(val_loss)
        val_accuracy = np.mean(val_accuracy)
        elapsed = time.time() - t0_epoch
        print(f"  [{name}] Epoch {epoch_i+1:>2} | train loss {avg_train_loss:.4f} | "
              f"val loss {val_loss:.4f} | val acc {val_accuracy:.2f}% | {elapsed:.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping at epoch {epoch_i+1}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def evaluate_on_test(model, test_dataloader, y_test):
    model.eval()
    preds_list = []
    for batch in test_dataloader:
        b_input_ids, b_attn_mask, _ = tuple(t.to(device) for t in batch)
        with torch.no_grad():
            logits = model(b_input_ids, b_attn_mask)
        preds_list.extend(torch.argmax(logits, dim=1).cpu().numpy())

    acc = accuracy_score(y_test, preds_list)
    report = classification_report(y_test, preds_list, target_names=sentiment, digits=3, output_dict=True)
    return acc, report, np.array(preds_list)


def class_restricted_mcnemar(preds_baseline, preds_attention, y_true, class_idx, class_name):
    mask = (y_true == class_idx)
    correct_b = (preds_baseline[mask] == y_true[mask])
    correct_a = (preds_attention[mask] == y_true[mask])

    n_both = np.sum(correct_b & correct_a)
    n_b_only = np.sum(correct_b & ~correct_a)
    n_a_only = np.sum(~correct_b & correct_a)
    n_neither = np.sum(~correct_b & ~correct_a)

    table = [[n_both, n_b_only], [n_a_only, n_neither]]
    result = mcnemar(table, exact=(n_b_only + n_a_only < 25), correction=True)

    print(f"  Class-restricted McNemar's test on '{class_name}' (n={mask.sum()}):")
    print(f"    Attention-only correct: {n_a_only} | Baseline-only correct: {n_b_only}")
    print(f"    p-value: {result.pvalue:.6f}")
    return result.pvalue, n_a_only, n_b_only

## 6. Run the full pipeline for each additional seed
Seeds 7 and 123 only (seed 2042 already validated in your original run: baseline other_cyberbullying F1=0.582, attention=0.636, class-restricted McNemar's p<0.0001).

In [ ]:
SEEDS = [7, 123]
EPOCHS = 10
all_results = []

for seed in SEEDS:
    print(f"\n{'='*70}\nSEED = {seed}\n{'='*70}")
    set_all_seeds(seed)

    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=0.2, stratify=y_all, random_state=seed)
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=seed)

    ros = RandomOverSampler(random_state=seed)
    X_train_res, y_train_res = ros.fit_resample(
        np.array(X_train).reshape(-1, 1), np.array(y_train).reshape(-1, 1))
    X_train = X_train_res.flatten()
    y_train = y_train_res.flatten()

    train_inputs, train_masks = bert_tokenizer(X_train)
    val_inputs, val_masks = bert_tokenizer(X_valid)
    test_inputs, test_masks = bert_tokenizer(X_test)

    train_labels = torch.tensor(y_train, dtype=torch.long)
    val_labels = torch.tensor(y_valid, dtype=torch.long)
    test_labels = torch.tensor(y_test, dtype=torch.long)

    batch_size = 32
    train_data = TensorDataset(train_inputs, train_masks, train_labels)
    train_dataloader = DataLoader(train_data, sampler=RandomSampler(train_data), batch_size=batch_size)
    val_data = TensorDataset(val_inputs, val_masks, val_labels)
    val_dataloader = DataLoader(val_data, sampler=SequentialSampler(val_data), batch_size=batch_size)
    test_data = TensorDataset(test_inputs, test_masks, test_labels)
    test_dataloader = DataLoader(test_data, sampler=SequentialSampler(test_data), batch_size=batch_size)

    seed_result = {"seed": seed}
    seed_preds = {}

    for model_class, key in [(Bert_Classifier_CLS, "baseline"), (Bert_Classifier_AttentionPool, "attention")]:
        model, optimizer, scheduler = initialize_model(model_class, train_dataloader, epochs=EPOCHS)
        model = bert_train(model, optimizer, scheduler, train_dataloader, val_dataloader,
                            epochs=EPOCHS, patience=3, name=f"{key} seed={seed}")
        acc, report, preds = evaluate_on_test(model, test_dataloader, y_test)

        seed_result[f"{key}_accuracy"] = acc
        seed_result[f"{key}_macro_f1"] = report["macro avg"]["f1-score"]
        seed_result[f"{key}_other_cb_recall"] = report["other_cyberbullying"]["recall"]
        seed_result[f"{key}_other_cb_f1"] = report["other_cyberbullying"]["f1-score"]
        seed_preds[key] = preds

        print(f"  [{key}] seed={seed} -> accuracy={acc:.4f}, "
              f"other_cyberbullying recall={report['other_cyberbullying']['recall']:.3f}")

        del model
        torch.cuda.empty_cache()

    p_value, n_attn_only, n_base_only = class_restricted_mcnemar(
        seed_preds["baseline"], seed_preds["attention"], np.array(y_test),
        TARGET_CLASS_IDX, "other_cyberbullying")
    seed_result["mcnemar_p_value"] = p_value
    seed_result["attention_only_correct"] = n_attn_only
    seed_result["baseline_only_correct"] = n_base_only

    all_results.append(seed_result)

print("\nAll additional seed runs complete.")

## 7. Combine with your original seed=2042 result

In [ ]:
original_2042 = {
    "seed": 2042,
    "baseline_accuracy": 0.8593,
    "attention_accuracy": 0.8602,
    "baseline_other_cb_recall": 0.523,
    "attention_other_cb_recall": 0.687,
    "baseline_other_cb_f1": 0.582,
    "attention_other_cb_f1": 0.636,
    "mcnemar_p_value": 0.0000,   # reported as < 0.0001
    "attention_only_correct": 217,
    "baseline_only_correct": 28,
}

combined_results = [original_2042] + all_results
results_df = pd.DataFrame(combined_results)
print("Per-seed results (all 3 seeds):")
print(results_df.to_string(index=False))
print()

print("other_cyberbullying recall across seeds:")
print(f"  Baseline:  mean={results_df['baseline_other_cb_recall'].mean():.4f}, "
      f"std={results_df['baseline_other_cb_recall'].std():.4f}")
print(f"  Attention: mean={results_df['attention_other_cb_recall'].mean():.4f}, "
      f"std={results_df['attention_other_cb_recall'].std():.4f}")
print()
print("Class-restricted McNemar's p-values across seeds:")
print(results_df[["seed", "mcnemar_p_value", "attention_only_correct", "baseline_only_correct"]].to_string(index=False))